# Collocation

In [2]:
import polars as pl
import polars_corpus as plc
import math

In [18]:
bnc = pl.scan_parquet("../bnc.parquet") #.filter(pl.col('mode')=='written')

In [19]:
m = plc.search_cqp(bnc, '[token="dangerous"]')

In [20]:
m.concordance(window=3)

token_left_context,token,token_right_context
list[str],list[str],list[str]
"[""it"", ""was"", ""too""]","[""dangerous""]","[""."", ""One"", ""of""]"
"[""away"", ""could"", ""be""]","[""dangerous""]","[""and"", ""could"", ""therefore""]"
"[""behind"", ""in"", ""the""]","[""dangerous""]","[""town"", ""at"", ""the""]"
"[""would"", ""be"", ""‘""]","[""dangerous""]","[""’"", ""for"", ""the""]"
"[""set"", ""of"", ""potentially""]","[""dangerous""]","[""encounters"", ""between"", ""the""]"
…,…,…
"[""where"", ""the"", ""most""]","[""dangerous""]","[""place"", ""in"", ""the""]"
"[""they"", ""can"", ""be""]","[""dangerous""]","[""."", ""How"", ""many""]"
"[""renders"", ""it"", ""—""]","[""dangerous""]","["","", ""or"", ""less""]"


In [21]:
 m.collocates("token", window=3)

collocate,freqs
str,struct[4]
"""act""","{9,32676,11010,112429158}"
"""the""","{1088,32676,5405646,112429158}"
"""present""","{7,32676,24334,112429158}"
"""'s""","{306,32676,780018,112429158}"
"""both""","{37,32676,59867,112429158}"
…,…
"""could""","{127,32676,156259,112429158}"
"""ideas""","{8,32676,10468,112429158}"
"""her""","{26,32676,287910,112429158}"


In [22]:
collocs = m.collocates("token", window=3).sort(
    by=pl.col("freqs").struct.field("f12"), descending=True
)
collocs

collocate,freqs
str,struct[4]
""".""","{1760,32676,4715138,112429158}"
""",""","{1501,32676,5017057,112429158}"
"""and""","{1230,32676,2506072,112429158}"
"""a""","{1115,32676,2036673,112429158}"
"""the""","{1088,32676,5405646,112429158}"
…,…
"""good""","{5,32676,72982,112429158}"
"""tendency""","{5,32676,2849,112429158}"
"""accidents""","{5,32676,1795,112429158}"


In [23]:
ll = (
    collocs.with_columns(LL=pl.col("freqs").corpus.loglik())
    .sort(by="LL", descending=True)
    .head(20)
)
ll

collocate,freqs,LL
str,struct[4],f64
"""potentially""","{154,32676,2373,112429158}",1370.04015
"""most""","{261,32676,87690,112429158}",745.686673
"""very""","{285,32676,114997,112429158}",721.001424
"""be""","{616,32676,647322,112429158}",611.472485
"""is""","{757,32676,971795,112429158}",550.752081
…,…,…
"""precedent""","{28,32676,705,112429158}",220.921203
"""are""","{321,32676,454599,112429158}",193.331588
"""situation""","{58,32676,15625,112429158}",188.834475


In [24]:
# double checking LL for 'potentially dangerous'

# a = frequency of node-collocate pairs
a = 154
# b = frequency of node without collocate
b = 32676 - a
# c = frequency of collocate without node
c = 2373 - a

# d = words in corpus - occurrences of node and collocate
N = 112429158
d = N - a - b - c

LL2 = 2 * (
    a * math.log(a)
    + b * math.log(b)
    + c * math.log(c)
    + d * math.log(d)
    - (a + b) * math.log(a + b)
    - (a + c) * math.log(a + c)
    - (b + d) * math.log(b + d)
    - (c + d) * math.log(c + d)
    + (N) * math.log(N)
)
LL2

1370.0401501655579

In [25]:
collocs = m.collocates(pl.struct(pl.col("token"),pl.col("pos")), window=3)

In [26]:
ll = (
    collocs.with_columns(LL=pl.col("freqs").corpus.loglik())
    .sort(by="LL", descending=True)
    .head(100)
)
ll

collocate,freqs,LL
struct[2],struct[4],f64
"{""potentially"",""ADV""}","{154,32676,2373,112429158}",1370.04015
"{""most"",""ADV""}","{256,32676,55442,112429158}",938.980975
"{""very"",""ADV""}","{284,32676,108814,112429158}",744.56278
"{""more"",""ADV""}","{305,32676,135968,112429158}",718.299974
"{""be"",""VERB""}","{616,32676,647141,112429158}",611.713342
…,…,…
"{""weapons"",""SUBST""}","{11,32676,3798,112429158}",30.816756
"{""risky"",""ADJ""}","{6,32676,618,112429158}",30.519775
"{""subversive"",""ADJ""}","{5,32676,309,112429158}",30.45416


In [27]:
ll.filter(pl.col('collocate').struct.field("pos")=="ADV")

collocate,freqs,LL
struct[2],struct[4],f64
"{""potentially"",""ADV""}","{154,32676,2373,112429158}",1370.04015
"{""most"",""ADV""}","{256,32676,55442,112429158}",938.980975
"{""very"",""ADV""}","{284,32676,108814,112429158}",744.56278
"{""more"",""ADV""}","{305,32676,135968,112429158}",718.299974
"{""too"",""ADV""}","{188,32676,64942,112429158}",527.352435
…,…,…
"{""inherently"",""ADV""}","{9,32676,415,112429158}",60.05796
"{""incredibly"",""ADV""}","{10,32676,706,112429158}",58.27803
"{""possibly"",""ADV""}","{18,32676,6547,112429158}",48.746851


In [28]:
ll.filter(pl.col('collocate').struct.field("pos")=="ADJ")

collocate,freqs,LL
struct[2],struct[4],f64
"{""driving"",""ADJ""}","{35,32676,1180,112429158}",255.537116
"{""difficult"",""ADJ""}","{39,32676,21421,112429158}",77.654571
"{""dangerous"",""ADJ""}","{22,32676,5445,112429158}",75.063668
"{""unpredictable"",""ADJ""}","{8,32676,660,112429158}",44.168506
"{""violent"",""ADJ""}","{12,32676,2602,112429158}",43.908422
…,…,…
"{""exciting"",""ADJ""}","{11,32676,3164,112429158}",34.472712
"{""unhealthy"",""ADJ""}","{5,32676,265,112429158}",31.978292
"{""risky"",""ADJ""}","{6,32676,618,112429158}",30.519775


In [29]:
ll.filter(pl.col('collocate').struct.field("pos")=="SUBST")

collocate,freqs,LL
struct[2],struct[4],f64
"{""substances"",""SUBST""}","{38,32676,1287,112429158}",277.095224
"{""precedent"",""SUBST""}","{28,32676,705,112429158}",220.921203
"{""situation"",""SUBST""}","{58,32676,15625,112429158}",188.834475
"{""driving"",""SUBST""}","{26,32676,1289,112429158}",169.757799
"{""chemicals"",""SUBST""}","{26,32676,1822,112429158}",151.916679
…,…,…
"{""sport"",""SUBST""}","{11,32676,3343,112429158}",33.363997
"{""criminals"",""SUBST""}","{7,32676,870,112429158}",33.051544
"{""species"",""SUBST""}","{16,32676,8820,112429158}",31.752817


time is money

In [45]:
m = plc.search(bnc, 'time')
collocs = m.collocates(pl.struct((pl.col('lemma'),pl.col('pos'))), window=5, min_freq=50)
time_ll = (
    collocs.with_columns(LL=pl.col("freqs").corpus.loglik())
    .filter(pl.col('freqs').struct.field("f12")>50)
    .sort(by="LL", descending=True)
    .with_row_index()
)

In [46]:
time_ll

index,collocate,freqs,LL
u32,struct[2],struct[4],f64
0,"{""at"",""PREP""}","{35129,1525870,521829,112429158}",58505.707545
1,"{""same"",""ADJ""}","{8290,1525870,61105,112429158}",24250.502063
2,"{""first"",""ADJ""}","{9365,1525870,120614,112429158}",17764.688153
3,"{""spend"",""VERB""}","{4459,1525870,22086,112429158}",16619.314117
4,"{""long"",""ADJ""}","{5063,1525870,40822,112429158}",13924.189634
…,…,…,…
1975,"{null,""STOP""}","{175593,1525870,13614355,112429158}",-533.526928
1976,"{""with"",""PREP""}","{6769,1525870,658931,112429158}",-587.79904
1977,"{""of"",""PREP""}","{36512,1525870,3041681,112429158}",-596.610205


In [47]:
m = plc.search(bnc, 'money')
collocs = m.collocates(pl.struct((pl.col('lemma'),pl.col('pos')))
                       , window=5, min_freq=50)
money_ll = (
    collocs.with_columns(LL=pl.col("freqs").corpus.loglik())
    .filter(pl.col('freqs').struct.field("f12")>50)
    .sort(by="LL", descending=True)
    .with_row_index()
)

In [48]:
collocs

collocate,freqs
struct[2],struct[4]
"{""rise"",""VERB""}","{70,365350,14938,112429158}"
"{""four"",""ADJ""}","{55,365350,45329,112429158}"
"{""anyone"",""PRON""}","{53,365350,14277,112429158}"
"{""sell"",""VERB""}","{150,365350,21038,112429158}"
"{""part"",""SUBST""}","{142,365350,64482,112429158}"
…,…
"{""small"",""ADJ""}","{152,365350,50362,112429158}"
"{""borrow"",""VERB""}","{328,365350,3000,112429158}"
"{""research"",""SUBST""}","{113,365350,25533,112429158}"


In [53]:
joint = time_ll.join(money_ll, on='collocate', how='inner').with_columns(((pl.col('index')+pl.col('index_right'))/2).alias('rank')).sort(by='rank', descending=False).select('rank','collocate').unnest()
joint

rank,lemma,pos
f64,str,str
1.5,"""spend""","""VERB"""
5.0,"""for""","""PREP"""
22.0,"""much""","""ADJ"""
24.5,"""have""","""VERB"""
26.0,"""waste""","""VERB"""
…,…,…
1285.5,"""'s""","""UNC"""
1288.5,"""group""","""SUBST"""
1290.0,"""as""","""PREP"""


In [54]:
joint.filter(pl.col('pos')=='VERB')

rank,lemma,pos
f64,str,str
1.5,"""spend""","""VERB"""
24.5,"""have""","""VERB"""
26.0,"""waste""","""VERB"""
28.5,"""save""","""VERB"""
31.5,"""get""","""VERB"""
…,…,…
1153.5,"""include""","""VERB"""
1166.0,"""suggest""","""VERB"""
1229.0,"""lead""","""VERB"""


In [55]:
joint.filter(pl.col('pos')=='SUBST')


rank,lemma,pos
f64,str,str
28.5,"""lot""","""SUBST"""
33.0,"""amount""","""SUBST"""
39.0,"""waste""","""SUBST"""
61.0,"""money""","""SUBST"""
64.0,"""time""","""SUBST"""
…,…,…
1265.5,"""state""","""SUBST"""
1266.5,"""area""","""SUBST"""
1280.0,"""number""","""SUBST"""


In [56]:
joint.filter(pl.col('pos')=='ADJ')


rank,lemma,pos
f64,str,str
22.0,"""much""","""ADJ"""
26.0,"""more""","""ADJ"""
27.5,"""some""","""ADJ"""
30.5,"""any""","""ADJ"""
37.5,"""that""","""ADJ"""
…,…,…
1212.5,"""2""","""ADJ"""
1215.5,"""general""","""ADJ"""
1232.0,"""those""","""ADJ"""
